In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models
from PIL import ImageFile
import pandas as pd
import numpy as np
import shutil
import os

ImageFile.LOAD_TRUNCATED_IMAGES = True
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [2]:
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

modelVGG = models.Sequential()
modelVGG.add(base_model)
modelVGG.add(layers.GlobalMaxPooling2D())
modelVGG.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 512)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# train + vali

In [4]:
trainValiPath='/content/gdrive/MyDrive/ml/train+vali'
trainValiImage = datagen.flow_from_directory(trainValiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)
trainValifeatures = modelVGG.predict(trainValiImage, verbose=1)

Found 2954 images belonging to 18 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


93/93 ━━━━━━━━━━━━━━━━━━━━ 1442s 16s/step


In [5]:
trainValifilenames = trainValiImage.filenames
trainValifeatures = trainValifeatures / np.linalg.norm(trainValifeatures, axis=1, keepdims=True)
df_trainVali = pd.DataFrame(trainValifeatures)
df_trainVali.insert(0, "filename", trainValifilenames)
df_trainVali.to_csv("trainValiFeature.csv", index=False)

In [6]:
df_trainVali.to_csv("trainValiFeature.csv", index=False)
shutil.move('trainValiFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/trainValiFeature.csv'

# seperate train and vali

In [7]:
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [8]:
trainImage = datagen.flow_from_directory(trainPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )
valiImage= datagen.flow_from_directory(valiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [9]:
trainLabel = []
valiLabel = []

for folder in sorted(os.listdir(trainPath)):
    folder_path = os.path.join(trainPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                trainLabel.append((label, label+"/"+file))

for folder in sorted(os.listdir(valiPath)):
    folder_path = os.path.join(valiPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                valiLabel.append((label, label+"/"+file))


In [10]:
train = pd.DataFrame(trainLabel, columns=['label', 'pathname'])
vali = pd.DataFrame(valiLabel, columns=['label', 'pathname'])

train.to_csv('trainLabel.csv', index=False)
vali.to_csv('valiLabel.csv', index=False)

In [11]:
train_features = modelVGG.predict(trainImage, verbose=1)
val_features = modelVGG.predict(valiImage, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 1284s 14s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 315s 14s/step


In [12]:
train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

train_features = train_features / np.linalg.norm(train_features, axis=1, keepdims=True)
val_features = val_features / np.linalg.norm(val_features, axis=1, keepdims=True)

df_train = pd.DataFrame(train_features)
df_val = pd.DataFrame(val_features)

df_train.insert(0, "filename", train_filenames)
df_val.insert(0, "filename", val_filenames)

df_train.to_csv("trainFeature.csv", index=False)
df_val.to_csv("valiFeature.csv", index=False)

In [13]:
import joblib

filename = 'vggFinetune.sav'
joblib.dump(modelVGG, filename)

['vggFinetune.sav']

In [16]:
shutil.move('vggFinetune.sav', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainLabel.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('valiLabel.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('valiFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/valiFeature.csv'

# End of VGG

In [17]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [18]:
trainFeature= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainFeature.csv')
valiFeature=pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [19]:
from sklearn.preprocessing import StandardScaler
stdscaler = StandardScaler()
stdscaler.fit(trainFeature)

trainFeature = stdscaler.transform(trainFeature)
valiFeature = stdscaler.transform(valiFeature)

In [20]:
trainLabel= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainLabel.csv')
valiLabel = pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/valiLabel.csv')

trainLabel=trainLabel.drop(['pathname'],axis=1)
valiLabel=valiLabel.drop(['pathname'],axis=1)

In [21]:
trainLabel=pd.get_dummies(trainLabel['label'])
valiLabel=pd.get_dummies(valiLabel['label'])
valiLabel=valiLabel.reindex(columns=trainLabel.columns, fill_value=0)

trainLabel = trainLabel.astype(int)
valiLabel = valiLabel.astype(int)

In [22]:
from tensorflow.keras.optimizers import Adam

In [23]:
model = models.Sequential()
model.add(layers.Dense(512, input_shape=(512,)))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(512))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(18, activation='softmax'))
model.summary()

adamm = Adam(learning_rate=0.0001)
model.compile(optimizer=adamm, loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 18)             │         9,234 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 538,642 (2.05 MB)

 Trainable params: 536,594 (2.05 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [24]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

In [25]:
history = model.fit(trainFeature, trainLabel, epochs=300,
                    validation_data=(valiFeature, valiLabel),
                    callbacks=[early_stop])

Epoch 1/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.2168 - loss: 2.6690 - val_accuracy: 0.7254 - val_loss: 1.3548
Epoch 2/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6974 - loss: 1.1981 - val_accuracy: 0.8791 - val_loss: 0.7254
Epoch 3/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8341 - loss: 0.7490 - val_accuracy: 0.9198 - val_loss: 0.4641
Epoch 4/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8877 - loss: 0.5154 - val_accuracy: 0.9396 - val_loss: 0.3325
Epoch 5/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9231 - loss: 0.4066 - val_accuracy: 0.9593 - val_loss: 0.2525
Epoch 6/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9456 - loss: 0.3051 - val_accuracy: 0.9698 - val_loss: 0.1976
Epoch 7/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9492 - loss: 0.2567 - val_accuracy: 0.9842 - val_loss: 0.1551
Epoch 8/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9539 - loss: 0.2390 - val_accuracy: 0.9882 - 

In [26]:
new_model = models.Sequential(model.layers[:-1])

for old_layer, new_layer in zip(model.layers[:-1], new_model.layers):
    new_layer.set_weights(old_layer.get_weights())
new_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,408 (2.02 MB)

 Trainable params: 527,360 (2.01 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [27]:
joblib.dump(new_model, 'cnnVGGFinetune.sav')

['cnnVGGFinetune.sav']

# Train done

In [28]:
import joblib
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [29]:
# stdscaler = joblib.load('/content/gdrive/MyDrive/ml/vggFinetune/stdScaler.pkl')
# new_model = joblib.load('/content/gdrive/MyDrive/ml/vggFinetune/cnnVGGFinetune.sav')

FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/ml/vggFinetune/stdScaler.pkl'

In [30]:
trainValiFeature= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainValiFeature.csv')
trainValiFilename=trainValiFeature['filename']
trainValiFeature=trainValiFeature.drop(['filename'],axis=1)

In [35]:
trainValiFeatures = stdscaler.transform(trainValiFeature.values)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [36]:
trainValiFeatures[0]

array([-8.90826170e-01, -4.59023217e-01,  4.02093583e-01, -4.88724194e-01,
       -1.07427053e+00,  9.79952334e-01, -1.96159991e-01, -8.85324846e-01,
       -3.89930875e-01,  1.70642927e+00, -5.00009408e-01, -5.23971370e-01,
        8.13407466e-02, -1.13888444e-01, -8.21950141e-01, -9.35332164e-01,
       -5.96118324e-01,  1.30825192e+00, -2.65902974e-01, -1.02490817e+00,
       -9.39779056e-01, -1.18241486e-01, -3.23430391e-02, -8.87543205e-01,
       -4.63727865e-01, -6.80121916e-01,  1.28606719e-02,  1.00928809e+00,
        1.08573097e+00,  4.28287330e-01, -1.24433552e+00,  7.74609210e-02,
        6.70703749e-01, -4.38447430e-01,  2.04643723e-01,  1.39762462e+00,
       -4.87604773e-01, -2.45587943e-01,  5.82066400e-01, -3.95959335e-01,
       -8.31575311e-01, -2.88918875e-01, -1.02748108e+00, -5.61638534e-01,
        4.80414282e-01, -5.26076063e-01,  7.01404986e-02,  8.75836678e-01,
       -4.79015757e-01,  1.08569429e-03, -1.84293290e-01, -6.94444394e-01,
       -4.41972526e-01, -

In [37]:
trainVali = new_model.predict(trainValiFeatures, verbose=1)

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [38]:
trainVali.shape

(2954, 512)

In [39]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(trainVali)

MinMaxScaler(feature_range=(-1, 1))

In [ ]:
trainVali[0]

In [40]:
trainVali = scaler.transform(trainVali)

In [42]:
trainVali[0]

array([-1.        , -1.        ,  0.40953028, -0.16522926, -1.        ,
       -1.        , -0.5039332 , -0.7568879 , -1.        , -1.        ,
       -0.74614495, -1.        , -0.25823057, -1.        , -1.        ,
        0.60339403, -1.        , -0.9664392 , -0.7334794 , -1.        ,
       -0.20406973, -1.        , -0.61097324, -1.        , -1.        ,
       -0.7427634 , -1.        ,  0.58299685, -1.        , -1.        ,
       -0.8532592 , -1.        , -1.        , -1.        , -0.49209905,
        0.42020857, -1.        , -1.        , -1.        , -1.        ,
       -1.        , -1.        , -1.        , -0.81952816,  0.31355417,
       -1.        , -0.8046411 , -1.        , -0.93816715, -0.017268  ,
       -0.8517355 , -1.        , -0.8341596 , -1.        , -1.        ,
        0.08984518, -1.        , -0.15660638, -0.8726751 , -1.        ,
       -1.        , -1.        , -1.        , -1.        , -0.16221905,
       -1.        , -1.        , -1.        , -0.7295637 , -0.88

In [43]:
df_trainVali = pd.DataFrame(trainVali)
df_trainVali.insert(0, "filename", trainValiFilename)

In [44]:
joblib.dump(scaler, 'minMaxScaler.pkl')

['minMaxScaler.pkl']

In [45]:
import joblib
df_trainVali.to_csv("trainValiVectors.csv", index=False)
joblib.dump(scaler, 'minMaxScaler.pkl')
joblib.dump(new_model, 'cnnVGGFinetune.sav')
joblib.dump(stdscaler, 'stdScaler.pkl')

['stdScaler.pkl']

In [46]:
shutil.move('cnnVGGFinetune.sav', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('minMaxScaler.pkl', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('stdScaler.pkl', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainValiVectors.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/trainValiVectors.csv'